# 01 - Ingestão de Dados

Camada **Bronze**: extração dos dados brutos da fonte original, sem nenhuma transformação, gravados em `dados/bronze/`.

In [1]:
import sys
from pathlib import Path

# Raiz do projeto (notebooks/ -> raiz do repositorio). Fixado aqui porque o
# kernel do Jupyter roda com cwd = pasta do notebook, nao a raiz do projeto.
RAIZ_PROJETO = Path.cwd().parent
if str(RAIZ_PROJETO) not in sys.path:
    sys.path.append(str(RAIZ_PROJETO))

from src.ingestao.ingestao import Ingestao

## Parametros da ingestao

Ajuste os anos e trimestres abaixo e rode a celula seguinte — nao e
preciso abrir o `.py` para mudar o periodo baixado.


In [2]:
# Anos da PNAD Continua a baixar
ANOS = (2023, 2024, 2025)

# Trimestres: 1 = Jan-Mar, 2 = Abr-Jun, 3 = Jul-Set, 4 = Out-Dez
TRIMESTRES = (1, 2, 3, 4)

In [ ]:
import logging

# Os logs INFO detalhados de cada download (retentativas, checksum etc.)
# continuam disponíveis via `python -m src.ingestao.ingestao` — aqui no
# notebook deixamos só avisos/erros, e a tabela-resumo da próxima célula
# concentra o que importa conferir visualmente.
logging.getLogger("src.ingestao.ingestao").setLevel(logging.WARNING)

etapa_ingestao = Ingestao(
    caminho_saida=RAIZ_PROJETO / "dados" / "bronze",
    anos=ANOS,
    trimestres=TRIMESTRES,
)
resultados_ingestao = etapa_ingestao.executar()
print(f"Ingestão concluída: {len(resultados_ingestao)} período(s) processado(s).")

## Resumo da ingestão

Tabela com o status de cada período (ano/trimestre) — muito mais rápida de
ler do que rolar o log bruto. `erro` aparece destacado; se houver algum, a
célula acima já teria levantado uma exceção antes de chegar aqui.

In [ ]:
import pandas as pd

CORES_STATUS = {
    "baixado": "color: #1a7f37; font-weight: bold",
    "substituido": "color: #9a6700; font-weight: bold",
    "sem_mudanca": "color: #57606a",
    "erro": "color: #cf222e; font-weight: bold",
}

resumo_ingestao = (
    pd.DataFrame([r.__dict__ for r in resultados_ingestao])
    .assign(periodo=lambda df: df["ano"].astype(str) + "Q" + df["trimestre"].astype(str))
    .set_index("periodo")[["status", "linhas", "duracao_segundos"]]
    .round({"duracao_segundos": 1})
)

resumo_ingestao.style.map(lambda s: CORES_STATUS.get(s, ""), subset=["status"])

## Conferencia rapida (head 10) da camada Bronze

Rode as celulas abaixo, uma de cada vez, para inspecionar visualmente o
arquivo bruto gerado pela ingestao (sem decodificacao — isso e trabalho do
Silver). Por padrao usa o ultimo periodo de ANOS/TRIMESTRES configurados
acima.


In [4]:
import json

ANO_CONFERENCIA = max(ANOS)
TRIMESTRE_CONFERENCIA = max(TRIMESTRES)

pasta = RAIZ_PROJETO / "dados" / "bronze" / str(ANO_CONFERENCIA)
caminho_txt = pasta / f"PNADC_0{TRIMESTRE_CONFERENCIA}{ANO_CONFERENCIA}.txt"
caminho_manifesto = pasta / f"PNADC_0{TRIMESTRE_CONFERENCIA}{ANO_CONFERENCIA}.manifest.json"

In [5]:
manifesto = json.loads(caminho_manifesto.read_text())
print(f"Arquivo: {caminho_txt}")
print(f"Total de linhas (manifesto): {manifesto['linhas']}")
print(f"Baixado em: {manifesto['load_timestamp']}")

Arquivo: C:\Users\Gui\OneDrive\Documentos\HandsOn -  Engenharia de Dados\pipeline-hands-on-engenharia-de-dados\dados\bronze\2025\PNADC_042025.txt
Total de linhas (manifesto): 498494
Baixado em: 2026-08-17T02:58:29.543027+00:00


In [6]:
N_LINHAS = 10
N_CARACTERES_PREVIA = 200  # cada linha real tem ~3480 caracteres (pesos de replicacao)

with open(caminho_txt, encoding="latin-1") as arquivo:
    for i, linha in enumerate(arquivo, start=1):
        linha = linha.rstrip(chr(10))
        print(f"{i:>2} | ({len(linha)} chars) {linha[:N_CARACTERES_PREVIA]}...")
        if i >= N_LINHAS:
            break

 1 | (3478 chars) 202541111  11000010811100120112511000288.26659886000349.37765480000519171003092024111217050101215061936089412            1  071  102  222222                                                            ...
 2 | (3478 chars) 202541111  11000010811100120112511000288.26659886000349.37765480000519171006375059111212050205218051968057412            1  10   103  122222                                                            ...
 3 | (3478 chars) 202541111  11000010811100120112511000288.26659886000349.37765480000519171006346334111111050305116041974051412            1  10   103  11              154143 80000 2 2             4      11  2   1 1130...
 4 | (3478 chars) 202541111  11000010811100120112511000288.26659886000349.37765480000519171007981109111108050410124071990035412            1  071  108  11              171233 23099 2 2             15     3 2 2   221130...
 5 | (3478 chars) 202541111  11000010811100120112511000288.26659886000349.37765480000519171007252363111102050510